In [8]:
import sqlite3
import os

os.makedirs("../data", exist_ok=True)
conn = sqlite3.connect("../data/vastu.db")
print("DB created!")
conn.close()


DB created!


In [10]:
from sqlalchemy import create_engine
from sqlalchemy.orm import declarative_base

engine = create_engine("sqlite:///../data/vastu.db")
Base = declarative_base()

print("Engine created!")

Engine created!


## Vastu Knowledge DB Schema

### Tables

| Table | Columns |
|-------|---------|
| directions | id, name, deity, element |
| rooms | id, name, aliases |
| pages | id, page_num, summary, full_text |
| index_entries | id, topic, category, keywords, page_refs, rule_count |
| rules | id, index_id(FK), page_id(FK), short_desc, full_detail, compatibility |
| rule_mappings | id, rule_id(FK), room_id(FK nullable), direction_id(FK nullable) |
| consequences | id, rule_id(FK), description, severity |

### Relationships
- `rules` → `index_entries` (many to one)
- `rules` → `pages` (many to one)
- `rule_mappings` → `rules` (many to one)
- `rule_mappings` → `rooms` (nullable)
- `rule_mappings` → `directions` (nullable)
- `consequences` → `rules` (many to one)

### Notes
- `index_entries` not `index` — index is a reserved word in SQL!
- `rule_mappings` nullables enable general rules (room-only or direction-only)
- `pages` has lazy loading — summary first, full_text on demand

In [13]:
from sqlalchemy import Column, Integer, String, ForeignKey
class Direction(Base):
    __tablename__ = "directions"
    __table_args__ = {'extend_existing': True}
    id = Column(Integer, primary_key=True)
    name = Column(String, nullable=False)
    deity = Column(String)
    element = Column(String)

class Room(Base):
    __tablename__ = "rooms"
    
    id = Column(Integer,primary_key=True)
    name = Column(String, nullable=False)
    aliases = Column(String)
    
class Page(Base):
    __tablename__ = "pages"
    
    id = Column(Integer,primary_key=True)
    page_num = Column(Integer,nullable=False)
    summary = Column(String(600))
    full_text = Column(String)
    
class IndexEntry(Base):
    __tablename__ = "index_entries"
    
    id = Column(Integer, primary_key=True)
    topic = Column(String,nullable=False)
    category = Column(String)
    keywords = Column(String)
    page_refs = Column(String)
    rule_count = Column(Integer)

class Rule(Base):
    __tablename__ ="rules"
    
    id = Column(Integer, primary_key=True)
    index_id = Column(Integer,ForeignKey("index_entries.id"))
    page_id = Column(Integer,ForeignKey("pages.id"))
    short_desc = Column(String(600))
    full_detail = Column(String)
    compatibility = Column(Integer)  # -2 (avoid) to 2 (auspicious)
    
class RuleMapping(Base):
    __tablename__ = "rules_mappings"
    
    id = Column(Integer, primary_key=True)
    rule_id = Column(Integer,ForeignKey("rules.id"))
    room_id = Column(Integer,ForeignKey("rooms.id"),nullable=True)
    direction_id = Column(Integer, ForeignKey("directions.id"), nullable=True)
    
class Consequence(Base):
    __tablename__ = "consequences"
    
    id = Column(Integer, primary_key=True)
    rule_id = Column(Integer, ForeignKey("rules.id"))
    description = Column(String(500))
    severity = Column(Integer)  # -2 (severe) to 2 (minor)

/var/folders/dd/_1d8w09n6njc4vzk9ysbtkvr0000gn/T/ipykernel_10174/3202927656.py:2: SAWarning: This declarative base already contains a class with the same class name and module name as __main__.Direction, and will be replaced in the string-lookup table.
  class Direction(Base):


In [14]:
Base.metadata.create_all(engine)
print("tables_created")


tables_created


In [15]:
from sqlalchemy import inspect

inspector = inspect(engine)
print(inspector.get_table_names())


['consequences', 'directions', 'index_entries', 'pages', 'rooms', 'rules', 'rules_mappings']
